# Chapter 2: First Query in 10 Minutes

This notebook contains all the code examples from Chapter 2.

## Setup

Make sure you have the required packages installed:
```bash
pip install duckdb polars pyarrow jupyter pandas
```

## Version Check

In [1]:
import duckdb
print(f'DuckDB {duckdb.__version__}')
# Should see 1.5.0 or higher

DuckDB 1.5.3


## Generate Sample Data (if needed)

Run this cell if you don't have `orders.parquet` yet:

In [2]:
import polars as pl
import datetime

# 10M orders across 2 years
orders = pl.DataFrame({
    'order_id': range(1, 10_000_001),
    'customer_id': (pl.arange(1, 10_000_001, eager=True) % 500_000) + 1,
    'order_date': pl.date_range(
        datetime.date(2022, 1, 1),
        datetime.date(2023, 12, 31),
        interval='1d',
        eager=True
    ).sample(10_000_000, with_replacement=True),
    'amount': (pl.arange(1, 10_000_001, eager=True) % 500 + 20.0),
    'status': pl.Series(['completed', 'pending', 'cancelled', 'refunded'])\
        .sample(10_000_000, with_replacement=True)
})

# Write as Parquet
orders.write_parquet('orders.parquet', compression='snappy')
print(f"[OK] Generated {len(orders):,} rows -> orders.parquet")

[OK] Generated 10,000,000 rows -> orders.parquet


## Your First Query

Query 10M rows of Parquet directly - no import, no loading:

In [3]:
import duckdb

# Query Parquet directly - no import, no loading
con = duckdb.connect()

result = con.execute("""
    SELECT
        DATE_TRUNC('month', order_date) as month,
        status,
        COUNT(*) as orders,
        SUM(amount) as revenue,
        AVG(amount) as avg_order
    FROM read_parquet('orders.parquet')
    WHERE order_date >= '2023-01-01'
    GROUP BY 1, 2
    ORDER BY 1, 2
""").fetchdf()

print(result)

        month     status  orders     revenue   avg_order
0  2023-01-01  cancelled  106171  28655521.0  269.899700
1  2023-01-01  completed  106077  28533803.0  268.991421
2  2023-01-01    pending  105700  28504692.0  269.675421
3  2023-01-01   refunded  106492  28710871.0  269.605895
4  2023-02-01  cancelled   96201  25885950.0  269.081922
5  2023-02-01  completed   95601  25686272.0  268.682043
6  2023-02-01    pending   96139  25982613.0  270.260903
7  2023-02-01   refunded   95604  25703875.0  268.857736
8  2023-03-01  cancelled  106215  28703285.0  270.237584
9  2023-03-01  completed  105795  28515849.0  269.538721
10 2023-03-01    pending  106053  28774101.0  271.318124
11 2023-03-01   refunded  106522  28712382.0  269.544151
12 2023-04-01  cancelled  102997  27787098.0  269.785508
13 2023-04-01  completed  102540  27614391.0  269.303599
14 2023-04-01    pending  102637  27667526.0  269.566784
15 2023-04-01   refunded  102499  27520792.0  268.498151
16 2023-05-01  cancelled  10585

## Check Compression Ratio

In [4]:
import os

file_size = os.path.getsize('orders.parquet') / 1_000_000
print(f"Compressed: {file_size:.0f} MB")
# Uncompressed in memory would be ~1.2 GB

Compressed: 101 MB


## Query Plan

See how DuckDB applies filters before loading data:

In [5]:
plan = con.execute("""
    EXPLAIN
    SELECT DATE_TRUNC('month', order_date) as month,
           COUNT(*) as orders
    FROM read_parquet('orders.parquet')
    WHERE order_date >= '2023-01-01'
    GROUP BY 1
""").fetchall()

# fetchall() returns (label, plan_text) tuples; print the plan text
print(plan[0][1])

┌───────────────────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│        Aggregates:        │
│        count_star()       │
│                           │
│      ~1,000,000 rows      │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│           month           │
│                           │
│      ~2,000,000 rows      │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│        READ_PARQUET       │
│    ────────────────────   │
│         Function:         │
│        READ_PARQUET       │
│                           │
│        Projections:       │
│         order_date        │
│                           │
│          Filters:         │
│ order_date>='2023-01-01': │
│           :DATE           │
│                           │
│      ~2,000,000 rows      │
└───────────────────────────┘



## Smoke Test Checklist

In [6]:
import duckdb
import polars as pl
import pyarrow.parquet as pq

# [OK] DuckDB can query Parquet
assert duckdb.execute("SELECT COUNT(*) FROM read_parquet('orders.parquet')").fetchone()[0] == 10_000_000
print("[OK] DuckDB can query Parquet")

# [OK] Polars can read Parquet lazily
df = pl.scan_parquet('orders.parquet')
assert df.select(pl.len()).collect().item() == 10_000_000
print("[OK] Polars can read Parquet lazily")

# [OK] PyArrow can read metadata without loading data
metadata = pq.read_metadata('orders.parquet')
print(f"[OK] PyArrow: {metadata.num_rows:,} rows in {metadata.num_row_groups} row groups")

print("\nStack verified. You're ready.")

[OK] DuckDB can query Parquet
[OK] Polars can read Parquet lazily
[OK] PyArrow: 10,000,000 rows in 82 row groups

Stack verified. You're ready.


## Quick Win Queries

### Top 10 Customers by Total Spend

In [7]:
top_customers = con.execute("""
    SELECT customer_id,
           SUM(amount) as total_spent,
           COUNT(*) as order_count
    FROM read_parquet('orders.parquet')
    GROUP BY customer_id
    ORDER BY total_spent DESC
    LIMIT 10
""").fetchdf()

print(top_customers)

   customer_id  total_spent  order_count
0       141000      10380.0           20
1       231500      10380.0           20
2       138500      10380.0           20
3       148000      10380.0           20
4       223000      10380.0           20
5       171000      10380.0           20
6       151000      10380.0           20
7       218500      10380.0           20
8       139000      10380.0           20
9       161000      10380.0           20


### Daily Order Volume with 7-Day Moving Average

In [8]:
daily_orders = con.execute("""
    SELECT order_date,
           COUNT(*) as orders,
           AVG(COUNT(*)) OVER (
               ORDER BY order_date
               ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
           ) as rolling_avg_7d
    FROM read_parquet('orders.parquet')
    GROUP BY order_date
    ORDER BY order_date
""").fetchdf()

print(daily_orders.head(10))

  order_date  orders  rolling_avg_7d
0 2022-01-01   13575    13575.000000
1 2022-01-02   13726    13650.500000
2 2022-01-03   13783    13694.666667
3 2022-01-04   13775    13714.750000
4 2022-01-05   13442    13660.200000
5 2022-01-06   13656    13659.500000
6 2022-01-07   13619    13653.714286
7 2022-01-08   13713    13673.428571
8 2022-01-09   13780    13681.142857
9 2022-01-10   13384    13624.142857


### Revenue by Status (Pie Chart Data)

In [9]:
revenue_by_status = con.execute("""
    SELECT status,
           SUM(amount) as revenue,
           ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) as pct
    FROM read_parquet('orders.parquet')
    GROUP BY status
    ORDER BY revenue DESC
""").fetchdf()

print(revenue_by_status)

      status      revenue    pct
0    pending  674451238.0  25.03
1   refunded  673701056.0  25.00
2  completed  673669671.0  25.00
3  cancelled  673178035.0  24.98
